# Parda — Phase 6, step 1b: compress GLiNER for the browser without losing accuracy

Step 1 showed: the ONNX conversion is perfect (100% the same spans as PyTorch), but plain 8-bit compression of
every layer kept only 36%. Here four careful compressions are tried, each checked against PyTorch on 100 real OCR
texts; the **smallest one that keeps ≥ 99%** is chosen and run through both full benchmarks.

**Settings:** Accelerator **None (CPU)** — free · Internet **On**
**Inputs:** Your Work → **Parda** (Phase 1) · Your Work → the **08b** notebook · Datasets tab → `celeba-dataset`
**Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`. The exported model comes from your HF repo `parda-onnx-v1` (made in step 1).

Run cells 1–4 by hand (~8 min), then **Save Version → Save & Run All** (~2 h).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to regenerate the unseen test pages) + GLiNER, YOLO, ONNX tools
!bash scripts/setup_kaggle.sh
!pip install -q gliner ultralytics onnx onnxruntime onnxconverter-common
!python -c "import gliner, onnx, onnxruntime, onnxconverter_common; print('gliner', gliner.__version__, '| onnxruntime', onnxruntime.__version__, '| onnxconverter-common ok')"


In [ ]:
# 3. Inputs + the full-precision ONNX export from step 1 (from your HF repo, ~1.2 GB)
import json, shutil
from huggingface_hub import HfApi, snapshot_download
from parda.fsutil import find_path
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import find_faces_dir
IN, W = "/kaggle/input", "/kaggle/working"
first = lambda *rels: next((h[0] for r in rels for h in [find_path(IN, r)] if h), None)
DATA = find_data()
FACES = find_faces_dir(IN, exclude=[DATA])
C_SEEN = first("cache_seen/ocr_easyocr.jsonl", "e2e_cache/ocr_easyocr.jsonl")
C_OOD = first("cache_ood/ocr_easyocr.jsonl", "ood_cache/ocr_easyocr.jsonl")
V2_SEEN, V2_OOD = first("v2_seen/report_e2e.json"), first("v2_ood/report_e2e.json")
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
ONNX_REPO = f"{HF_USER}/parda-onnx-v1"
snapshot_download(ONNX_REPO, local_dir=f"{W}/onnx", token=os.environ["HF_TOKEN"],
                  allow_patterns=["gliner/model_fp32.onnx", "gliner/parda_onnx.json", "gliner/tokenizer/*",
                                  "gliner/gliner_config.json", "yolo/*"])
meta_path = f"{W}/onnx/gliner/parda_onnx.json"
meta = json.load(open(meta_path))
meta["files"] = {"fp32": meta["files"]["fp32"]}  # the plain 8-bit file is not used any more
meta.pop("default", None)
json.dump(meta, open(meta_path, "w"), indent=2)
for k, v in [("DATA", DATA), ("FACES", FACES), ("OCR seen", C_SEEN), ("OCR unseen", C_OOD), ("v2 report seen", V2_SEEN),
             ("v2 report unseen", V2_OOD), ("fp32 model", f"{os.path.getsize(f'{W}/onnx/gliner/model_fp32.onnx') / 2**20:.0f} MB"),
             ("YOLO onnx", os.path.exists(f"{W}/onnx/yolo/parda-yolo.onnx"))]:
    print(f"{k:18s} {v}")
assert DATA and _n_images(DATA) == 30000 and FACES and C_SEEN and C_OOD and V2_SEEN and V2_OOD
C_SEEN, C_OOD = os.path.dirname(C_SEEN), os.path.dirname(C_OOD)


In [ ]:
# 4. Tests (~2 min): every line ok, none "skipped"
!python tests/test_export.py 2>&1 | grep -E "^ok|skipped|Error|assert"


In [ ]:
# 5. COMPRESS: 4 variants, each checked against PyTorch on 100 real OCR texts (~35 min)
!python -m parda.export.compress --gliner_dir {W}/onnx/gliner --parity_ocr {C_SEEN}/ocr_tesseract.jsonl {C_OOD}/ocr_easyocr.jsonl --parity_n 50 2>&1 | grep -v -i warning | grep -E "MB|could not|chosen"
cs = json.load(open(f"{W}/onnx/gliner/compress_summary.json"))
CHOSEN = cs["chosen"]
print("\nchosen:", CHOSEN, "| good enough:", cs["chosen_passed"], "| fp32:", cs["fp32_mb"], "MB")


In [ ]:
# 6. END TO END with the chosen model: seen templates (OCR reused) (~20 min)
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
os.makedirs(f"{W}/cache_seen", exist_ok=True)
for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
    shutil.copy(f"{C_SEEN}/{f}", f"{W}/cache_seen/{f}")
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_seen --out {W}/c_seen --title "Compressed GLiNER ({CHOSEN}) — seen templates" 2>&1 | grep -v -i warning | tail -9


In [ ]:
# 7. END TO END with the chosen model: unseen layouts + phone photos (~20 min)
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
os.makedirs(f"{W}/cache_ood", exist_ok=True)
for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
    shutil.copy(f"{C_OOD}/{f}", f"{W}/cache_ood/{f}")
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_ood --out {W}/c_ood --title "Compressed GLiNER ({CHOSEN}) — unseen layouts + phone photos" 2>&1 | grep -v -i warning | tail -9


In [ ]:
# 8. PyTorch v2 vs the chosen compressed model, side by side
from IPython.display import Markdown, display
pct = lambda v: "–" if v is None else f"{100 * v:.1f}%"
rows = [f"| Benchmark | Configuration | All PII: PyTorch → {CHOSEN} | Text | Pages with no leak | s/page (CPU) |", "|---|---|---|---|---|---|"]
drops = []
for bench, v2p, cp in (("seen", V2_SEEN, f"{W}/c_seen/report_e2e.json"), ("unseen", V2_OOD, f"{W}/c_ood/report_e2e.json")):
    v2 = {r["config"]: r for r in json.load(open(v2p))["configs"]}
    cc = {r["config"]: r for r in json.load(open(cp))["configs"]}
    for name in ("+ YOLO visual = Parda", "browser build: Tesseract only + YOLO"):
        a, b = v2.get(name, {}), cc[name]
        if a:
            drops.append(a["fully_redacted"] - b["fully_redacted"])
        rows.append(f"| {bench} | {name} | {pct(a.get('fully_redacted'))} → **{pct(b['fully_redacted'])}** | "
                    f"{pct(a.get('text'))} → {pct(b['text'])} | {pct(a.get('pages_clean'))} → {pct(b['pages_clean'])} | {b['sec_per_page']:.2f} |")
var = ["| Variant | Size | Same spans as PyTorch |", "|---|---|---|"] + [
    f"| {k} | {v.get('mb', '–')} MB | {pct(v.get('agreement_f1'))} |" if "error" not in v else f"| {k} | – | not made: {v['error'][:60]} |"
    for k, v in cs["variants"].items()]
report = "\n".join([f"### Compressed GLiNER for the browser: chosen **{CHOSEN}**", "",
                    f"fp32 ONNX: {cs['fp32_mb']} MB, 100% the same spans as PyTorch (step 1). Plain 8-bit of every layer: 36.2%.", "",
                    *var, "", *rows, "", f"Largest drop vs PyTorch: {100 * max(drops):.1f} points." if drops else ""]) + "\n"
open(f"{W}/compressed_vs_pytorch.md", "w").write(report)
display(Markdown(report))


In [ ]:
# 9. Save the chosen model + its record to the HF repo
api = HfApi(token=os.environ["HF_TOKEN"])
g = f"{W}/onnx/gliner"
for f in [meta_path, f"{g}/compress_summary.json"] + ([f"{g}/model_{CHOSEN}.onnx"] if CHOSEN != "fp32" else []):
    api.upload_file(path_or_fileobj=f, path_in_repo=f"gliner/{os.path.basename(f)}", repo_id=ONNX_REPO)
print("uploaded", CHOSEN, "to", f"https://huggingface.co/{ONNX_REPO}")


In [ ]:
# 10. Collect results to commit to GitHub (download from the Output tab)
R = f"{W}/compress_results"
os.makedirs(R, exist_ok=True)
for src, dst in [(f"{W}/compressed_vs_pytorch.md", "compressed_vs_pytorch.md"), (f"{g}/compress_summary.json", "compress_summary.json"),
                 (f"{W}/c_seen/report_e2e.md", "report_seen.md"), (f"{W}/c_seen/report_e2e.json", "report_seen.json"),
                 (f"{W}/c_ood/report_e2e.md", "report_unseen.md"), (f"{W}/c_ood/report_e2e.json", "report_unseen.json")]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(R, dst))
print(sorted(os.listdir(R)))
